In [27]:
import os
import numpy as np
from dataclasses import dataclass
from pathlib import Path

In [28]:
#!pip install numpy 

In [29]:
@dataclass
class CreateVectorsConfig:
    root_dir : Path
    source_path : str 
    local_data_file : str

In [30]:
os.chdir("c:\\Users\\RSR\\PYTHON\\LLMIssueTracker\src")

In [31]:
from LLMIssueTracker.Utils.Common import *
from LLMIssueTracker.Entity.Config_Entity import *
from LLMIssueTracker.Constant import * 

In [54]:
class ConfigurationManager:
    def __init__(self,
                 config_path = CONFIG_FILE_PATH,
                 params_path =PARAMS_FILE_PATH
                 ):
        os.chdir(r"C:\Users\RSR\PYTHON\LLMIssueTracker")
        self.config=Read_Yaml(Path(config_path))
        self.params=Read_Yaml(Path(params_path))
        
        self.Create_Vectors=self.config.Create_Vectors

    def get_create_vectors_config(self) -> CreateVectorsConfig:
        config=self.config.Create_Vectors
        create_directories([config.root_dir])
        create_vectors_config=CreateVectorsConfig(
            root_dir=config.root_dir,
            source_path=config.source_path,
            local_data_file=config.local_data_file
        )
        return create_vectors_config

In [55]:
import pandas as pd 
from sentence_transformers import SentenceTransformer
import faiss
import pickle

In [56]:
#!pip install faiss-cpu

In [57]:
class CreateVectors:
    def __init__(self,
                 config : CreateVectorsConfig
                 ):
        self.config=config
    def Create_Vectors(self):
        model =SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"    
        )
        df =pd.read_csv(self.config.source_path)
        docs=[]

        for _,row in df.iterrows():
            docs.append(f"""
             Issue: {row['COMMENTS']}
             Root Cause: {row['PROCESS']}
             Resolution: {row['LOG_TEXT']}
             """
             )
        vectors=model.encode(docs)
        dimentions =vectors.shape[1]
        index=faiss.IndexFlatL2(dimentions)
        index.add(vectors)

        faiss.write_index(
            index,
            self.config.local_data_file
        )

        with open(
            self.config.local_data_file,
            "wb"
        ) as f:
            pickle.dump(docs,f)

        print("FAISS index created")

        

In [58]:
try:
    config_manager = ConfigurationManager()

    config = config_manager.get_create_vectors_config()

    vectors = CreateVectors(config)

    vector = vectors.Create_Vectors()

except Exception as e:
    raise e

Reading YAML : Config\Config.yaml
[2026-07-01 20:00:00,901: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-07-01 20:00:00,908: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-07-01 20:00:00,911: INFO: Common: Created directory : artifacts/Create_Vectors]
[2026-07-01 20:00:00,918: INFO: model: No device provided, using cpu]
[2026-07-01 20:00:01,194: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-01 20:00:01,207: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"]
[2026-07-01 20:00:01,461: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4268.36it/s]


[2026-07-01 20:00:03,312: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-01 20:00:03,542: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-01 20:00:03,776: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-01 20:00:04,008: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-01 20:00:04,239: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-01 20:00:04,251: INFO: _client: HTTP Request: HEAD h

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.39it/s]

FAISS index created
